# Notebook Overview

This Notebook attempted to increase the diversity and size of the dataset through data augmentation. Though it failed to several crtical reasons which ill explain below. I decided to abort all operations since the process was time consuming and computationally expensive. I reattempted the same approach on another notebook though refined.

The notebook experienced challenges with both slow performance and low accuracy. Here are the key contributing factors:

Reasons for Low Accuracy:

- Double Normalization: The most critical issue was the redundant normalization of image data. The images were first scaled by manually dividing by 255.0, and then again by the ImageDataGenerator's rescale=1./255 parameter. This resulted in pixel values becoming excessively small, effectively hindering the model's ability to learn.
- Unnormalized Validation Data: Initially, the validation dataset (test_images) was not subjected to the same normalization procedures as the training data, leading to a significant mismatch in data distribution during evaluation. This skewed validation metrics and provided an - - - inaccurate representation of model performance.
- Incorrect Output Layer Size: The model's final Dense layer was configured for 100 output classes, while the dataset being used was CIFAR-10, which has only 10 classes. This mismatch prevented the model from making correct predictions.

# Reasons for Slow Performance:

- zca_whitening=True: The inclusion of ZCA whitening, a computationally intensive preprocessing technique, significantly increased the time required for datagen.fit() and added overhead to the batch generation process.
- samplewise_center and samplewise_std_normalization: These operations, which compute and apply statistics for each individual image within every batch, introduced considerable computational overhead during data loading.
- Excessive and Complex Augmentations: A wide range of augmentations, including brightness_range, channel_shift_range, and potentially inappropriate transformations like vertical_flip for certain image types, contributed to increased processing time per batch.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

In [2]:
#loading original dataset
cifar = tf.keras.datasets.cifar10

In [3]:
(train_images, train_labels), (test_images, test_labels) = cifar.load_data()

In [4]:
# Normalizing data
train_images, test_images = train_images / 255.0, test_images / 255.0

In [5]:
train_images.shape

(50000, 32, 32, 3)

# Perfoming Data Augmentation

In [6]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator



In [16]:
datagen = ImageDataGenerator(featurewise_center=True,  # Set input mean to 0 over the dataset
    samplewise_center=True,  # Set each sample mean to 0
    featurewise_std_normalization=True,  # Divide inputs by std of the dataset
    samplewise_std_normalization=True,  # Divide each input by its std
    zca_whitening=True,  # Apply ZCA whitening
    zca_epsilon=1e-06,  # Epsilon for ZCA whitening
    rotation_range=20,  # Degree range for random rotations
    width_shift_range=0.1,  # Fraction of total width for horizontal shifts
    height_shift_range=0.1,  # Fraction of total height for vertical shifts
    brightness_range=[0.5, 1.5],  # Range for randomly adjusting brightness
    shear_range=0.1,  # Shear intensity (shear angle in degrees)
    zoom_range=0.2,  # Range for random zoom
    channel_shift_range=0.1,  # Range for random channel shifts
    fill_mode='nearest',  # Strategy for filling in new pixels after transformations
    cval=0.0,  # Value used for points outside the boundaries
    horizontal_flip=True,  # Randomly flip inputs horizontally
    vertical_flip=True,  # Randomly flip inputs vertically

    preprocessing_function=None,  # Function applied to each image
    data_format=None,  # Image data format (channels_first or channels_last)
    validation_split=0.2,  # Fraction of data to reserve for validation
    interpolation_order=1,  # Order of interpolation for transformations
    dtype=None  # Desired data type for output
)

# The train_images are already in the correct format (NumPy array with batch dimension)
# for ImageDataGenerator, so img_to_array is not needed and causes an error.

''''
calculates the statistics (like the mean and standard deviation for featurewise_center and featurewise_std_normalization, or the principal components for zca_whitening) from your train_images dataset.
It then stores these statistics internally, so that when it generates augmented images later, it applies the normalization consistently based on the original training data's distribution. This prevents data leakage from the test set or new images.
fit() learns how to transform, and flow() applies those transformations (along with random augmentations) in batches during training.
'''
datagen.fit(train_images)
'''
This iterator will be responsible for:
Taking your train_images and train_labels.
Dividing them into batches of 32 images (as specified by batch_size=32).
Applying the random data augmentations (rotations, shifts, flips, brightness changes, etc.) that you configured in your datagen object to each image within those batches.
Yielding (providing) these augmented batches of images and their corresponding labels whenever requested by the training loop.
'''
train_generator = datagen.flow(train_images, train_labels, batch_size =32)

/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:1054: UserWarning: This ImageDataGenerator specifies `zca_whitening` which overrides setting of`featurewise_std_normalization`.
  warnings.warn(


In [8]:
shape = train_images[0].shape

# Creation of convolution Network that will train on Augmented Data

In [9]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
model = tf.keras.Sequential()
Conv_layer1 = Conv2D(32, (3,3),activation ='relu',input_shape = shape) # First filter application
Max_pool1 = MaxPooling2D(2,2) # First pooled layer
Conv_layer2 = Conv2D(64, (3,3), activation = 'relu')#second filter application
Max_pool2 = MaxPooling2D(2,2) # Second pooled layer
Conv_layer3 = Conv2D(64,(3,3), activation = 'relu') #3rd filter application

#Adding Layers to sequential layer
model.add(Conv_layer1)
model.add(Max_pool1)
model.add(Conv_layer2)
model.add(Max_pool2)
model.add(Conv_layer3)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
#Visiualizing model parameters

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,320 (220.00 KB)

 Trainable params: 56,320 (220.00 KB)

 Non-trainable params: 0 (0.00 B)

# Building Dense Network
Dense Network Makes Final Predictions

In [11]:
model.add(Flatten())
model.add(Dense(64, activation = 'relu'))
model.add(Dense(10, activation= 'softmax'))

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,570 (478.79 KB)

 Trainable params: 122,570 (478.79 KB)

 Non-trainable params: 0 (0.00 B)

# Fitting dataset to generate 100,000 augmented images (double the initial dataset)

In [13]:
batch_size = 32
double_epoch = 2*(train_images.shape[0] // batch_size)

In [14]:
model.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy',
              metrics = ['accuracy'])

# Training Dataset with Augmented Images

In [18]:
model.fit(train_generator,
        steps_per_epoch = double_epoch,# it stops the iterator once 100k augmented images are generated, n 3125 batches
        epochs = 20, # Corrected 'epoch' to 'epochs'
        validation_data= (test_images, test_labels)) #fulldataset will be processed 20 times

Epoch 1/20
  10/3124 ━━━━━━━━━━━━━━━━━━━━ 14:00 270ms/step - accuracy: 0.1057 - loss: 2.3022

KeyboardInterrupt: 